# GRADIEND: A System Demonstration using the Example of English Pronouns

This notebook walks through [GRADIEND](https://arxiv.org/abs/2502.01406) (Drechsel & Herbold, ICLR 2026) using English pronouns as the running example. 
GRADIEND is a method for **learning features within neural networks** by training an encoder–decoder on model gradients; with it you can **find where a language model encodes a feature** (e.g. grammatical number, person) and **rewrite the model** to strengthen or weaken that feature while keeping other behaviour.

![GRADIEND overview](../../docs/img/workflow-diagram.png)

**Requires:** `pip install gradiend[recommended]`

## What is GRADIEND?

GRADIEND is a method for **learning features within neural networks** by training an encoder–decoder architecture on model gradients. With this library you can **find where a language model encodes a feature** (e.g. gender, grammatical number, person) and **rewrite the model** to strengthen or weaken it—for example to debias it—while keeping other behaviour.

GRADIEND works by:
1. **Training an encoder–decoder network** on gradients computed from masked text predictions
2. **Learning a single latent feature neuron** that encodes the desired interpretation (e.g. 3SG vs 3PL)
3. **Using the decoder** to modify the base model’s weights, enabling targeted feature manipulation


---

## Running example: English pronouns (grammatical number and person)

We use **English personal pronouns** as in the paper and the [start tutorial](https://aieng-lab.github.io/gradiend/start/): the feature is grammatical **number** (singular vs plural) or **person** (1st vs 2nd vs 3rd). 
For each sentence we mask the pronoun (e.g. *The nurse Mary said that **[MASK]** was exhausted*), define the **factual** fill-in (e.g. *she*) and the **counterfactual** (e.g. *they*). The base model’s gradients on these two alternatives define a direction in parameter space; the data-generation step produces the texts and labels needed for that.

| Person\Number | Singular | Plural |
|---------------|----------|--------|
| 1st           | I        | we     |
| 2nd           | you      | you    |
| 3rd           | he/she/it| they   |




In the following, we follow the standard GRADIEND pipeline: 

- **Feature Selection & Data Generation**
- **GRADIEND training**
- **Intra-Model Evaluation** (encoder and decoder) 
- **Model Rewrite**
- **Inter-Model Evaluation**. 

## Step 1: Feature selection and data generation

To extract the desired feature, we need data where feature-related tokens are **masked** and each example is labelled by the fill-in (factual). The package provides `TextPredictionDataCreator`: it takes raw texts and **feature targets** (one per pronoun class), masks those tokens, and produces per-class training data. We also create **neutral** sentences (no target pronouns) for later evaluation.

In [ ]:
from gradiend import TextFilterConfig, TextPredictionDataCreator, TextPreprocessConfig

creator = TextPredictionDataCreator(
    base_data="wikipedia", # load English Wikipedia from HuggingFace
    hf_config="20220301.en",
    # base_data="path/to/my/data.csv" # or load from a custom CSV (see docs)
    # base_data=["Sentence 1, Sentence 2, ..."] # or load from a list of strings
    preprocess=TextPreprocessConfig(min_chars=20, max_chars=200),
    feature_targets=[
        TextFilterConfig(targets=["he", "she", "it"], id="3SG"),
        TextFilterConfig(target="they", id="3PL"),
    ],
)

training = creator.generate_training_data(max_size_per_class=1000)
neutral = creator.generate_neutral_data(additional_excluded_words=["his", "him", "her", "them"], max_size=5000)

In [ ]:
training.keys() # dict of class_id -> pandas DataFrame

In [ ]:
training["3SG"].head()

In [ ]:
neutral.head() # texts do not contain any pronouns!

---

## Step 2: GRADIEND training

With the generated data, we train a **GRADIEND model**: an encoder–decoder network on the **gradient differences** between factual and counterfactual fill-ins. 
The **encoder** maps these gradient differences to a single scalar (the “latent feature neuron”, e.g. −1 for one class and +1 for the other). 
The **decoder** maps that scalar to a parameter-update direction in the base model. Training thus learns where and how the model encodes the feature (see [Training](https://aieng-lab.github.io/gradiend/tutorials/training/) and the paper). 

Below we train on **3SG vs 3PL** only; the same data will later be used for all pronoun pairs and merged groups in Step 5.
The `gradiend` package provides a HuggingFace Trainer-like API with config and training arguments. For more details on the options, see the docs and the paper.

In [ ]:
from gradiend import TextPredictionConfig

config = TextPredictionConfig(
    run_id="3SG_3PL",
    data=training,
    target_classes=["3SG", "3PL"],
    eval_neutral_data=neutral,
)

In [1]:
from gradiend import TrainingArguments, PrePruneConfig, PostPruneConfig
args = TrainingArguments(
    experiment_dir="runs/notebook_english_pronouns",
    train_batch_size=16,
    train_max_size=20000,
    eval_steps=250,
    max_steps=1000,
    encoder_eval_max_size=50,
    decoder_eval_max_size_training_like=100,
    decoder_eval_max_size_neutral=500,
    num_train_epochs=3,
    max_seeds=5,
    min_convergent_seeds=1,
    source="alternative",
    target="diff",
    eval_batch_size=8,
    learning_rate=1e-5,
    pre_prune_config=PrePruneConfig(n_samples=16, topk=0.01, source="diff"),
    post_prune_config=PostPruneConfig(topk=0.05, part="decoder-weight"),
    add_identity_for_other_classes=False,
    use_cache=True,
)

model_name = "bert-base-cased"

In [ ]:
from gradiend import TextPredictionTrainer
trainer = TextPredictionTrainer(model=model_name, config=config, args=args)
trainer.train()

In [ ]:
trainer.plot_training_convergence()

---

## Step 3: Intra-model evaluation

**Intra-model** evaluation analyzes the quality of the **single** GRADIEND model: 

(1) **Encoder** — Do the encoded gradients separate the two feature classes? The trainer runs evaluation (and optionally neutral) data through the encoder and computes a **correlation** between encoded values and class labels; high correlation means the encoder has learned to distinguish the classes. We also get a distribution plot: target classes should sit at opposite ends (±1) and neutral near 0.

(2) **Decoder** — Can we change the base model’s behaviour by applying the learned direction? Decoder evaluation tries different learning rates and feature factors and selects a configuration that shifts probabilities toward the target class while satisfying a language-model constraint (LMS)

See [Evaluation intra-model](https://aieng-lab.github.io/gradiend/tutorials/evaluation-intra-model/) for more details.

In [ ]:
enc_eval = trainer.evaluate_encoder(plot=True)
print("Encoder correlation (training):", enc_eval.get("correlation"))
print("Means by class:", enc_eval.get("means_by_class"))

In [ ]:
dec = trainer.evaluate_decoder(plot=True, target_class="3SG")
print(dec["3SG"])

---

## Step 4: Model rewrite

Based on the learned feature direction of GRADIEND, we can create models with controlled changed feature behavior.

We can control two parameters: feature factor (the scalar input of the decoder) and the learning rate (the scale of the update). 
As it is difficult to pick these parameters manually, the decoder evaluation computes best parameters to get maximum feature change while keeping near base model performance on other tokens (LMS constraint).

We **apply** the decoder-selected update (learning rate and feature factor) to the base model’s weights to obtain a modified model biased toward the chosen class (e.g. 3SG). This is the “Model rewrite” step in the workflow; the result can be used in memory or saved to disk (see [Model rewrite](https://aieng-lab.github.io/gradiend/tutorials/model-rewrite/)).

In [ ]:
changed_model = trainer.rewrite_base_model(decoder_results=dec, target_class="3SG")
# changed_model is the base model with weights updated along the 3SG direction (strengthen).
# To save: trainer.rewrite_base_model(..., output_dir="path/to/rewritten")

---

## Step 5: Inter-model evaluation (English pronoun family)

So far, we only dealt with a *single* GRADIEND model. Now we want to compare *multiple* GRADIEND models, and therefore compare how different features are encoded in the base model.

The full English pronoun family has 5 classes (1SG, 1PL, 2SGPL, 3SG, 3PL) that differ in person and number, yielding 20 pairs.
Moreover, 


In [3]:
from gradiend import TextFilterConfig, TextPredictionDataCreator, TextPreprocessConfig

spacy_tags = {"pos": "PRON"} # we use spacy filtering now to minimize data noise
# Generate now the full data (with all 5 classes)
creator = TextPredictionDataCreator(
    base_data="wikipedia", # load English Wikipedia from HuggingFace
    hf_config="20220301.en",
    # base_data="path/to/my/data.csv" # or load from a custom CSV (see docs)
    # base_data=["Sentence 1, Sentence 2, ..."] # or load from a list of strings
    preprocess=TextPreprocessConfig(min_chars=20, max_chars=200),
    spacy_model="en_core_web_sm",
    feature_targets=[
        TextFilterConfig(target="I", id="1SG", spacy_tags=spacy_tags),
        TextFilterConfig(target="we", id="1PL", spacy_tags=spacy_tags),
        TextFilterConfig(target="you", id="2SGPL", spacy_tags=spacy_tags),
        TextFilterConfig(targets=["he", "she", "it"], id="3SG", spacy_tags=spacy_tags),
        TextFilterConfig(target="they", id="3PL", spacy_tags=spacy_tags),
    ],
    output_dir="data/notebook_english_pronouns",
    use_cache=True,
)

training_full = creator.generate_training_data(max_size_per_class=1000)

2026-02-26 18:23:40 - INFO - Using cached training data from data/notebook_english_pronouns/training.csv


In [ ]:
# --- Step 5.2: Training (all 10 pronoun pairs + 4 merged groups) ---
# Data: reuse training and neutral from Step 1 (all 5 classes).
from gradiend import TextPredictionTrainer, TextPredictionConfig

PRONOUN_CLASSES = ["1SG", "1PL", "2SGPL", "3SG", "3PL"]
pronoun_pairs = [
    (PRONOUN_CLASSES[i], PRONOUN_CLASSES[j])
    for i in range(len(PRONOUN_CLASSES))
    for j in range(i + 1, len(PRONOUN_CLASSES))
]
# Merged groups (see experiments/multilingual_gradiend_demo.py): Number SG↔PL; Person 1vs2, 1vs3, 2vs3
pronoun_merged_configs = [
    ("pronoun_number_singular_plural", {"singular": ["1SG", "3SG"], "plural": ["1PL", "3PL"]}, None),
    ("pronoun_person_1vs2", {"1st": ["1SG", "1PL"], "2nd": ["2SGPL"]}, None),
    ("pronoun_person_1vs3", {"1st": ["1SG", "1PL"], "3rd": ["3SG", "3PL"]}, [["1SG", "3SG"], ["1PL", "3PL"]]), # only 1SG<->1PL and 3SG<->3PL transitions allowed (not 1SG<->3PL) to encourage person-based encoding rather than number-based side effects.
    ("pronoun_person_2vs3", {"2nd": ["2SGPL"], "3rd": ["3SG", "3PL"]}, None),
]

models_pronoun_family = {}
# Train all 10 pairs
for c1, c2 in pronoun_pairs:
    run_id = f"pronoun_{c1}_{c2}"
    cfg = TextPredictionConfig(run_id=run_id, data=training_full, target_classes=[c1, c2])
    t = TextPredictionTrainer(model=model_name, config=cfg, args=args)
    t.train()
    t.cpu()
    models_pronoun_family[run_id] = t.get_model()

# Train 4 merged runs
for run_id_prefix, class_merge_map, transition_groups in pronoun_merged_configs:
    args.learning_rate = 1e-6
    kwargs = {"class_merge_map": class_merge_map, "data": training_full}
    if transition_groups is not None:
        kwargs["class_merge_transition_groups"] = transition_groups
    cfg = TextPredictionConfig(run_id=run_id_prefix, **kwargs)
    t = TextPredictionTrainer(model=model_name, config=cfg, args=args)
    t.train()
    t.cpu()
    models_pronoun_family[run_id_prefix] = t.get_model()

print("Trained", len(models_pronoun_family), "pronoun-family runs (10 pairs + 4 merged).")

2026-02-26 18:23:44 - INFO - GRADIEND model already exists at runs/notebook_english_pronouns/pronoun_1SG_1PL/model, skipping training. Use use_cache=False to retrain.
2026-02-26 18:23:44 - INFO - Inferred decoder eval targets for pronoun_1SG_1PL: {'1SG': ['i', 'I'], '1PL': ['We', 'WE', 'we']}


Some weights of the model checkpoint at google-bert/bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


2026-02-26 18:23:46 - INFO - GRADIEND model already exists at runs/notebook_english_pronouns/pronoun_1SG_2SGPL/model, skipping training. Use use_cache=False to retrain.
2026-02-26 18:23:47 - INFO - Inferred decoder eval targets for pronoun_1SG_2SGPL: {'1SG': ['i', 'I'], '2SGPL': ['You', 'YOU', 'you']}


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


2026-02-26 18:23:49 - INFO - GRADIEND model already exists at runs/notebook_english_pronouns/pronoun_1SG_3SG/model, skipping training. Use use_cache=False to retrain.
2026-02-26 18:23:49 - INFO - Inferred decoder eval targets for pronoun_1SG_3SG: {'1SG': ['i', 'I'], '3SG': ['it', 'It', 'SHE', 'he', 'HE', 'He', 'she', 'She', 'IT']}


...

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


2026-02-26 18:24:04 - INFO - Excluded 5 non-backbone (head) parameter(s); names: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
2026-02-26 18:24:04 - INFO - Set feature_class_encoding_direction: {'singular': 1, 'plural': -1}
2026-02-26 18:24:23 - INFO - Training GRADIEND model over 1,077,197 base model weights with 1 feature neurons
2026-02-26 18:24:23 - INFO - Output: runs/notebook_english_pronouns/pronoun_number_singular_plural/seeds/seed_0
2026-02-26 18:24:23 - INFO - Running initial evaluation before training...
2026-02-26 18:24:27 - INFO - Saved encoder metrics to runs/notebook_english_pronouns/pronoun_number_singular_plural/encoded_values.json
2026-02-26 18:24:27 - INFO - Step 0, Correlation: -0.0712, mean: plural: -0.0143, singular: -0.0230


Epoch 1/3:  31%|███       | 249/800 [00:43<01:36,  5.72it/s]

2026-02-26 18:25:16 - INFO - Saved encoder metrics to runs/notebook_english_pronouns/pronoun_number_singular_plural/encoded_values.json
2026-02-26 18:25:16 - INFO - Step 250, Correlation: 0.0523, mean: plural: -0.6752, singular: -0.6471


Epoch 1/3:  36%|███▋      | 290/800 [00:55<01:29,  5.71it/s]

In [ ]:
# --- Step 5.3: Plotting (Venn 3-run, Venn 6-run, heatmap subset) --- 
from pathlib import Path
from gradiend import plot_topk_overlap_heatmap, plot_topk_overlap_venn

topk = 1000
out_dir = Path(args.experiment_dir)
out_dir.mkdir(parents=True, exist_ok=True)

# Pretty labels for display (run_id -> axis label)
def pretty_label(rid):
    if rid == "pronoun_3SG_3PL":
        return "3SG ↔ 3PL"
    if rid.startswith("pronoun_number_"):
        return "SG ↔ PL"
    if rid.startswith("pronoun_person_"):
        p = rid.replace("pronoun_person_", "")
        return {"1vs2": "1st↔2nd", "1vs3": "1st↔3rd", "2vs3": "2nd↔3rd"}.get(p, p)
    if rid.startswith("pronoun_"):
        parts = rid.replace("pronoun_", "").split("_")
        return " ↔ ".join(parts) if len(parts) == 2 else rid
    return rid

models_display = {pretty_label(rid): model for rid, model in models_pronoun_family.items()}

In [ ]:
# Venn: 3-run subset (e.g. 1SG–3PL, 1SG–3SG, 3SG–3PL)
venn3_ids = ["pronoun_1SG_3PL", "pronoun_1SG_3SG", "pronoun_3SG_3PL"]
venn3_ids = [x for x in venn3_ids if x in models_pronoun_family]
venn3_models = {pretty_label(id): models_pronoun_family[id] for id in venn3_ids}
plot_topk_overlap_venn(venn3_models, topk=topk, part="decoder-weight", output_path=str(out_dir / "topk_overlap_venn_3_pronouns.png"))

In [ ]:
# Venn: 6-run subset
venn6_ids = ["pronoun_1SG_1PL", "pronoun_1SG_2", "pronoun_1SG_3SG", "pronoun_1SG_3PL", "pronoun_3SG_3PL", "pronoun_number_singular_plural"]
venn6_ids = [x for x in venn6_ids if x in models_pronoun_family]
venn6_models = {pretty_label(id): models_pronoun_family[id] for id in venn6_ids}
plot_topk_overlap_venn(venn6_models, topk=topk, part="decoder-weight", output_path=str(out_dir / "topk_overlap_venn_6_pronouns.png"))

In [ ]:
# Heatmap: all runs
heatmap_models = {pretty_label(id): models_pronoun_family[id] for id in models_pronoun_family}
plot_topk_overlap_heatmap(heatmap_models, topk=topk, part="decoder-weight", value="intersection_frac", output_path=str(out_dir / "topk_overlap_heatmap_pronoun_family.png"))

---

## Next steps

- **[Start here](https://aieng-lab.github.io/gradiend/start/)** — Minimal workflow with artificial texts.
- **[Detailed workflow](https://aieng-lab.github.io/gradiend/tutorials/detailed-workflow/)** — How the five steps connect; precomputed vs generated data.
- **[Data generation](https://aieng-lab.github.io/gradiend/tutorials/data-generation/)** — Syncretism, spaCy, morphology for richer features.
- **[Evaluation (intra-model)](https://aieng-lab.github.io/gradiend/tutorials/evaluation-intra-model/)** — Encoder/decoder options and metrics.
- **[Evaluation (inter-model)](https://aieng-lab.github.io/gradiend/tutorials/evaluation-inter-model/)** — Top-k overlap and heatmaps in full.
- **[Model rewrite](https://aieng-lab.github.io/gradiend/tutorials/model-rewrite/)** — Saving and loading rewritten checkpoints.

See also: [Examples](https://aieng-lab.github.io/gradiend/examples/) and [API reference](https://aieng-lab.github.io/gradiend/api/).